# Notebook 04 — Clasificación multiclase MNIST con TensorFlow + Keras (MLP, sin CNN)

**Curso:** Deep Learning – Conceptos · Especialización en Inteligencia Artificial · Universidad de Cundinamarca
**Semana 3:** clasificación multiclase con TensorFlow + Keras.

Arquitectura (idéntica a la del Notebook 05 en PyTorch):

| Capa | Operación | Salida | Parámetros |
|---|---|---|---|
| Entrada | imagen 28×28 | (28, 28) | 0 |
| `Flatten` | aplanado 28×28 → 784 | (784) | 0 |
| `Dense` + `ReLU` | $784 \times 256$ | (256) | 200 960 |
| `Dense` + `ReLU` | $256 \times 128$ | (128) | 32 896 |
| `Dense` + `Softmax` | $128 \times 10$ | (10) | 1 290 |

**Restricción:** no se usa ninguna capa convolucional (`Conv2D`); todas las capas son densas.

> En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)* acelera el entrenamiento,
> aunque este MLP también entrena en CPU en pocos minutos.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Juanpatula2361/Taller-Redes-Neuronales-II/blob/main/notebooks/04_mnist_mlp_keras.ipynb)

In [ ]:
# ============================================================
# 0. Importaciones, versiones y semillas
# ============================================================
import os
import time
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
keras.utils.set_random_seed(SEED)       # fija semillas de Python, NumPy y TensorFlow
FIG_DIR = "figuras"
os.makedirs(FIG_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

## 1. Carga y preprocesamiento de MNIST

- 60 000 imágenes de entrenamiento y 10 000 de prueba, en escala de grises de 28×28 píxeles (valores 0–255).
- **Normalización:** se divide entre 255 para llevar los píxeles a $[0, 1]$; esto estabiliza la magnitud de las
  activaciones y de los gradientes.
- **Aplanado:** el MLP necesita vectores, así que cada imagen $28\times 28$ se convierte en un vector de 784
  componentes. Se hace dentro del modelo con la capa `Flatten` (abajo se muestra la operación equivalente con `reshape`).
- **Codificación de etiquetas:** con `sparse_categorical_crossentropy` las etiquetas se usan como **enteros**
  0–9; la versión *one-hot* (necesaria para `categorical_crossentropy`) se muestra como referencia.
- **Validación:** se reservan las últimas 6 000 imágenes de entrenamiento (`validation_split=0.1`).

In [ ]:
# ============================================================
# 1. Carga, normalización, aplanado y codificación de etiquetas
# ============================================================
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print("Formas originales:", x_train.shape, y_train.shape, x_test.shape, y_test.shape)
print("Rango de píxeles original:", x_train.min(), "-", x_train.max())

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Rango tras normalizar:", x_train.min(), "-", x_train.max())

# Aplanado explícito (solo ilustrativo; el modelo lo hace con la capa Flatten)
x_train_flat = x_train.reshape(-1, 28 * 28)
print("Aplanado:", x_train.shape, "->", x_train_flat.shape)

# Codificación one-hot (referencia): la etiqueta 5 -> [0,0,0,0,0,1,0,0,0,0]
y_train_onehot = keras.utils.to_categorical(y_train, num_classes=10)
print(f"Etiqueta entera: {y_train[0]} -> one-hot: {y_train_onehot[0].astype(int)}")
print("Distribución de clases (train):", np.bincount(y_train))

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(x_train[i], cmap="gray"); ax.set_title(f"y = {y_train[i]}"); ax.axis("off")
plt.suptitle("Muestras de MNIST con su etiqueta real"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_muestras_mnist.png", dpi=150); plt.show()

## 2. Construcción del modelo (`Sequential`)

In [ ]:
# ============================================================
# 2. Modelo MLP: Flatten -> Dense+ReLU -> Dense+ReLU -> Dense+Softmax
# ============================================================
modelo = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(name="aplanado"),                        # 28x28 -> 784
    layers.Dense(256, name="densa_1"),                      # pesos: Glorot uniforme, sesgos: 0
    layers.ReLU(name="relu_1"),
    layers.Dense(128, name="densa_2"),
    layers.ReLU(name="relu_2"),
    layers.Dense(10, activation="softmax", name="salida"),  # distribución de probabilidad sobre 10 dígitos
], name="mlp_mnist_keras")

modelo.summary()

## 3. Compilación

- **Optimizador:** Adam ($\eta = 10^{-3}$, $\beta_1 = 0.9$, $\beta_2 = 0.999$).
- **Pérdida:** `sparse_categorical_crossentropy` $\;\mathcal{L} = -\frac{1}{m}\sum_i \log \hat{p}_{i,\,y_i}$.
- **Métrica:** accuracy.

In [ ]:
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 4. Entrenamiento (15 épocas)

`model.fit` ejecuta internamente, por cada lote: forward, cálculo de la pérdida, backpropagation
(diferenciación automática con `tf.GradientTape`) y el paso del optimizador. Un *callback* mide el tiempo de
cada época.

In [ ]:
class TiempoPorEpoca(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.tiempos = []

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.tiempos.append(time.perf_counter() - self._t0)


EPOCHS, BATCH_SIZE = 15, 128
cronometro = TiempoPorEpoca()
historial = modelo.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,     # últimas 6 000 imágenes como validación
    callbacks=[cronometro],
    verbose=2,
)
print(f"\nTiempo medio por época: {np.mean(cronometro.tiempos):.2f} s "
      f"(primera época: {cronometro.tiempos[0]:.2f} s, incluye compilación del grafo)")

## 5. Curvas de pérdida y precisión vs. épocas

In [ ]:
h = historial.history
ep = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, h["loss"], "o-", label="Entrenamiento"); axes[0].plot(ep, h["val_loss"], "s--", label="Validación")
axes[0].set_title("Keras: pérdida (entropía cruzada)"); axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss")
axes[1].plot(ep, h["accuracy"], "o-", label="Entrenamiento"); axes[1].plot(ep, h["val_accuracy"], "s--", label="Validación")
axes[1].set_title("Keras: accuracy"); axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy")
for ax in axes:
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/04_keras_curvas.png", dpi=150); plt.show()

## 6. Evaluación en el conjunto de prueba

In [ ]:
test_loss, test_acc = modelo.evaluate(x_test, y_test, verbose=0)
print(f"Keras -> test loss: {test_loss:.4f} | test accuracy: {test_acc:.4f}")

probs = modelo.predict(x_test, verbose=0)       # (10000, 10) probabilidades Softmax
y_pred = probs.argmax(axis=1)                   # clase con mayor probabilidad
print("\nReporte de clasificación por dígito:\n")
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Keras MLP – matriz de confusión (test acc = {test_acc:.2%})")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/04_keras_matriz_confusion.png", dpi=150); plt.show()

# Pares de dígitos que más se confunden (fuera de la diagonal)
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
top = np.dstack(np.unravel_index(np.argsort(cm_off.ravel())[::-1][:5], cm.shape))[0]
print("Confusiones más frecuentes (real -> predicho: casos):")
for r, p in top:
    print(f"  {r} -> {p}: {cm[r, p]}")

## 7. Aciertos y errores del clasificador

In [ ]:
errores = np.where(y_pred != y_test)[0]
aciertos = np.where(y_pred == y_test)[0]
fig, axes = plt.subplots(2, 6, figsize=(12, 5.4))
for ax, i in zip(axes[0], aciertos[:6]):
    ax.imshow(x_test[i], cmap="gray"); ax.axis("off")
    ax.set_title(f"real {y_test[i]} | pred {y_pred[i]}\np = {probs[i].max():.2f}", color="green", fontsize=9)
for ax, i in zip(axes[1], errores[:6]):
    ax.imshow(x_test[i], cmap="gray"); ax.axis("off")
    ax.set_title(f"real {y_test[i]} | pred {y_pred[i]}\np = {probs[i].max():.2f}", color="red", fontsize=9)
plt.suptitle(f"Fila superior: aciertos · Fila inferior: errores ({len(errores)} de {len(y_test)})")
plt.tight_layout(h_pad=2.0); plt.savefig(f"{FIG_DIR}/04_keras_aciertos_errores.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# 8. Guardado de métricas para la tabla comparativa
# ============================================================
metricas = {
    "framework": "TensorFlow/Keras",
    "version": tf.__version__,
    "parametros": int(modelo.count_params()),
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "train_loss_final": float(h["loss"][-1]),
    "val_loss_final": float(h["val_loss"][-1]),
    "val_accuracy_final": float(h["val_accuracy"][-1]),
    "segundos_por_epoca": float(np.mean(cronometro.tiempos)),
    "segundos_por_epoca_sin_primera": float(np.mean(cronometro.tiempos[1:])),
    "historial": {k: [float(v) for v in vals] for k, vals in h.items()},
    "matriz_confusion": cm.tolist(),
    "reporte": classification_report(y_test, y_pred, digits=4, output_dict=True),
}
with open("metricas_keras.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2)
print("Métricas guardadas en metricas_keras.json")